In [1]:
# 获取fusion candles

from research.model_pick.candle_fetch import FusionCandles

# 设定训练集范围
START = "2024-06-01"
END = "2025-06-01"

candle_container = FusionCandles(
    exchange="Binance Perpetual Futures", symbol="BTC-USDT", timeframe="1m"
)
candles = candle_container.get_candles(START, END)
candles.shape

PyTorch configured: device=cpu, dtype=torch.float32


/opt/homebrew/Caskroom/miniforge/base/envs/jesse/lib/python3.12/site-packages/jesse/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


(11545, 6)

In [2]:
# 制作标签
from research.model_pick.labeler import PipelineLabeler

LOG_RETURN_LAG = 4
LABEL_TYPE = "hard"

labeler = PipelineLabeler(candles, LOG_RETURN_LAG)

if LABEL_TYPE == "hard":
    # 分类模型标签
    raw_label = labeler.label_hard
else:
    # 回归模型标签
    raw_label = labeler.label_direction

raw_label.shape

(11541,)

In [3]:
# 制作特征
from research.model_pick.features import ALL_FEATS
from src.features.pipeline import FeatureMaker, FeatureMakerConfig

ALL_FEATS = ALL_FEATS[:100] # 简化以节约演示时间

feature_maker_config = FeatureMakerConfig(
    feature_names=ALL_FEATS,
    ssm_state_dim=5,
    verbose=True,
)

feature_maker = FeatureMaker(feature_maker_config)
feature_result = feature_maker.fit_transform(candles)
global_features = feature_result.features
print(global_features.shape)
global_features.head(1)

FeatureMaker: Starting fit_transform...
  [1/3] Computing raw features...
[100/100] wq_alpha_020: 0.000s | Memory: 4.40 MBK[K
  [2/3] Handling NaN values...
  Handling NaN values...
    First valid row: 376 (of 11545)
  [3/3] Training SSM and computing features...
  Training SSM models...
    No SSM configured, skipping...
FeatureMaker: fit_transform complete! Output shape: (11545, 100)
(11545, 100)


,frac_o_o1_diff,frac_o_o2_diff,frac_o_o3_diff,frac_o_o4_diff,frac_o_o5_diff,frac_o_h1_diff,frac_o_h2_diff,frac_o_h3_diff,frac_o_h4_diff,frac_o_h5_diff,...,wq_alpha_011,wq_alpha_012,wq_alpha_013,wq_alpha_014,wq_alpha_015,wq_alpha_016,wq_alpha_017,wq_alpha_018,wq_alpha_019,wq_alpha_020
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# 对齐label与feature，label开头会缺少LOG_RETURN_LAG个值，feature需要去掉开头的NaN

PRED_NEXT = 3

# 截断feature开头
features = global_features.iloc[LOG_RETURN_LAG:]

# shift以对齐PRED_NEXT
features = features.iloc[:-PRED_NEXT]
label = raw_label[PRED_NEXT:]

# 去掉feature开头的NaN
na_mask = features.isna().any(axis=1).values
features = features.iloc[~na_mask]
label = label[~na_mask]

assert len(features) == len(label)
print(len(label))

11166


In [7]:
# 特征筛选
from src.features.feature_selection import GrootCVConfig, GrootCVSelector

GROOTCV_CUTOFF = 3

groot_cv_config = GrootCVConfig(cutoff=GROOTCV_CUTOFF)
selector = GrootCVSelector(config=groot_cv_config, verbose=True)
selector.fit(features, label)

selected_features = selector.selected_features_
print(selected_features)

-> 识别数值型变量...
-> 任务类型: binary
-> 使用 GrootCV 进行特征选择 (cutoff=3.0)...

[OK] 特征选择完成：从 100 个特征中选择了 56 个，舍弃了 44 个 (cutoff=3.0)
['frac_c_c1_diff', 'frac_c_c4_diff', 'frac_c_c5_diff', 'frac_c_h1_diff', 'frac_c_h2_diff', 'frac_c_h3_diff', 'frac_c_h4_diff', 'frac_c_h5_diff', 'frac_c_l1_diff', 'frac_c_l2_diff', 'frac_c_l3_diff', 'frac_c_l5_diff', 'frac_c_o1_diff', 'frac_c_o2_diff', 'frac_c_o3_diff', 'frac_c_o4_diff', 'frac_c_o5_diff', 'frac_h_c1_diff', 'frac_h_h4_diff', 'frac_h_h5_diff', 'frac_h_l1_diff', 'frac_h_l2_diff', 'frac_h_l5_diff', 'frac_h_o5_diff', 'frac_l_c1_diff', 'frac_l_c3_diff', 'frac_l_h3_diff', 'frac_l_h4_diff', 'frac_l_l2_diff', 'frac_l_l3_diff', 'frac_l_l4_diff', 'frac_l_l5_diff', 'frac_l_o4_diff', 'frac_o_c1_diff', 'frac_o_l2_diff', 'frac_o_l3_diff', 'frac_o_l5_diff', 'frac_o_o2_diff', 'frac_o_o3_diff', 'frac_o_o4_diff', 'frac_o_o5_diff', 'wq_alpha_001', 'wq_alpha_002', 'wq_alpha_003', 'wq_alpha_004', 'wq_alpha_005', 'wq_alpha_007', 'wq_alpha_008', 'wq_alpha_009', 'wq_alpha_01